# 11 — Explanations for Compliance

Regulatory frameworks increasingly require algorithmic explanations. This notebook maps XAI methods to compliance requirements.

## EU AI Act and GDPR Requirements

**GDPR Article 22**: Data subjects have the right not to be subject to solely automated decisions. They can request 'meaningful information about the logic involved' and contest the decision. This requires:
- *Local explanations*: why was *this* person denied?
- *Counterfactual recourse*: what would need to change for a different decision?
- *Feature importance*: which factors drove the outcome?

**EU AI Act (2024)**: High-risk AI systems (credit, hiring, healthcare) must provide:
- *Transparency*: technical documentation of the model and training data
- *Human oversight*: mechanisms for human review of outputs
- *Explainability*: explanations legible to the affected person (not just the operator)
- *Audit logs*: records of decisions for post-hoc review

**Model Cards** (Mitchell et al., 2019): standardised documentation covering intended use, performance across demographic groups, ethical considerations, and known limitations. Recommended by Google and increasingly required by platforms.

**Compliance-ready explanation** means: correct, stable, locally accurate, actionable, and aligned with regulatory language (no jargon, no black-box references).

In [ ]:
# Compliance-ready explanation report generator
import numpy as np
from datetime import datetime
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
import json

np.random.seed(42)

# Simulate credit scoring model
feature_names = ['annual_income', 'credit_score', 'debt_to_income',
                 'employment_length', 'loan_amount', 'num_credit_lines']
feature_descriptions = {
    'annual_income': 'Annual income in GBP',
    'credit_score': 'Credit bureau score (300-850)',
    'debt_to_income': 'Monthly debt payments / monthly income',
    'employment_length': 'Years in current employment',
    'loan_amount': 'Requested loan amount in GBP',
    'num_credit_lines': 'Number of open credit lines',
}

X, y = make_classification(n_samples=1000, n_features=6, n_informative=4,
                            n_redundant=1, random_state=42)
scaler = StandardScaler()
X_sc = scaler.fit_transform(X)

model = GradientBoostingClassifier(n_estimators=100, random_state=42)
model.fit(X_sc[:800], y[:800])

print(f'Model accuracy: {model.score(X_sc[800:], y[800:]):.3f}')

In [ ]:
# Generate a compliance report for a denied applicant
def generate_compliance_report(model, x_raw, x_scaled, decision,
                                feature_names, feature_descriptions,
                                scaler, applicant_id='APP-001'):
    """
    Generate a GDPR-compliant explanation report.
    """
    # Feature importances from model
    importances = model.feature_importances_

    # Get top factors
    sorted_idx = np.argsort(importances)[::-1]

    # Build report
    report = {
        'report_id': f'EXPL-{applicant_id}-{datetime.now().strftime("%Y%m%d")}',
        'generated_at': datetime.now().isoformat(),
        'applicant_id': applicant_id,
        'decision': 'DECLINED' if decision == 0 else 'APPROVED',
        'approval_probability': float(model.predict_proba(x_scaled.reshape(1,-1))[0,1]),
        'model_version': 'credit-gbm-v2.1',
        'explanation': {
            'summary': '',
            'top_factors': [],
            'recourse': [],
        },
        'regulatory_basis': 'GDPR Article 22; EU AI Act Article 13',
        'appeal_instructions': 'Contact lending@example.com to request human review within 30 days.',
    }

    # Human-readable factor descriptions
    factors = []
    for idx in sorted_idx[:3]:
        fname = feature_names[idx]
        raw_val = x_raw[idx]
        imp = importances[idx]
        direction = 'negatively' if x_scaled[idx] < 0 else 'positively'
        factors.append({
            'factor': feature_descriptions[fname],
            'your_value': f'{raw_val:.2f}',
            'impact': f'{imp:.1%} of decision weight',
            'direction': direction,
        })
    report['explanation']['top_factors'] = factors

    if decision == 0:
        report['explanation']['summary'] = (
            'Your application was declined. The primary factors were '
            + ', '.join([f['factor'] for f in factors[:2]])
            + '. See recourse steps below to understand what could change this decision.'
        )
        report['explanation']['recourse'] = [
            f'Improving your {feature_descriptions[feature_names[sorted_idx[0]]]} '
             'may positively affect a future application.',
            'Reducing existing debt commitments before reapplying may improve your outcome.',
        ]

    return report

# Find a denied applicant
denied_mask = model.predict(X_sc[800:]) == 0
denied_idx = np.where(denied_mask)[0][0]
x_raw = X[800 + denied_idx]
x_scaled = X_sc[800 + denied_idx]
decision = 0

report = generate_compliance_report(
    model, x_raw, x_scaled, decision,
    feature_names, feature_descriptions, scaler
)

print(json.dumps(report, indent=2))

In [ ]:
# Model card template
model_card = {
    'model_name': 'CreditDecisionGBM v2.1',
    'intended_use': 'Automated credit application pre-screening for personal loans under £25,000',
    'out_of_scope': 'Business loans, mortgages, decisions for individuals under 18',
    'training_data': {
        'source': 'Historical loan applications 2018-2023',
        'size': '120,000 applications',
        'date_range': '2018-01 to 2023-12',
        'known_biases': 'Under-representation of applicants from Northern Ireland',
    },
    'performance': {
        'overall_accuracy': '0.847',
        'false_negative_rate': '0.12',
        'by_demographic': {
            'age_18_30': {'accuracy': '0.831', 'fpr': '0.14'},
            'age_31_50': {'accuracy': '0.856', 'fpr': '0.11'},
            'age_51_plus': {'accuracy': '0.849', 'fpr': '0.12'},
        },
    },
    'ethical_considerations': [
        'Model does not use protected characteristics (race, gender, religion) as features',
        'Annual fairness audit conducted by independent third party',
        'Human review available for all declined applications on request',
    ],
    'limitations': [
        'Performance may degrade for applicants with non-traditional income patterns',
        'Not validated for applicants outside the UK',
    ],
    'eu_ai_act_classification': 'High-risk (Annex III, credit scoring)',
    'last_updated': '2026-01-01',
}

print('=== MODEL CARD ===')
print(json.dumps(model_card, indent=2))